# Tutorial 01 — eXo-brain Core Framework

**What this notebook covers:**
- How the framework is structured and why it was built this way
- Running a complete single-turn orchestration with a deterministic tool call
- Running a multi-node background DAG with full observability evidence

**No API key required.** Everything runs in-process with in-memory adapters.

In [ ]:
# Load .env so any credentials / config are available in this session
import pathlib, os
_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
_env  = _root / ".env"
if _env.exists():
    try:
        from dotenv import load_dotenv
        load_dotenv(_env, override=False)
        print(f"✓ .env loaded from {_env}")
    except ImportError:
        print("⚠ python-dotenv not installed — install it with: pip install python-dotenv")
else:
    print(f"ℹ no .env found at {_env} — using system environment only")

## How eXo-brain works

eXo-brain is a **policy-governed execution layer** that sits between an AI model and its tools.

The central insight is: **model tool calls are intent, not execution.**
When a model says "call `delete_record(id=42)`", the framework intercepts that intent,
runs it through policy gates, and only then executes it — deterministically, with audit trail.

```
Host / API / CLI
      │
      ▼
OrchestratorHostAdapter       ← thin transport boundary
      │
      ▼
Orchestrator.run_turn()       ← provider-neutral turn loop
      │
      ├── RuntimeAdapter      ← pluggable provider (OpenAI, Ollama, custom…)
      │     └── yields RuntimeEvents (TOOL_INTENT, OUTPUT_DELTA, RUN_COMPLETE)
      │
      ├── PolicyMiddleware    ← before_tool_call: ALLOW / DENY / ESCALATE
      │     └── RiskGatePolicy, RBAC, tenant overlays
      │
      ├── ModeSelector        ← DETERMINISTIC or PROVIDER_NATIVE
      │     └── state-changing / HIGH/CRITICAL → always DETERMINISTIC
      │
      └── DeterministicToolExecutor
            └── validate → authz → retry → audit_log → redact → handler()
```

**Key guarantee:** state-changing or high-risk tool calls are *always* executed
deterministically, regardless of what the adapter or model requested.

## Section 1 — Single-Turn Orchestration

We will:
1. Register a tool in the `ToolRegistry`
2. Wire up the `Orchestrator` with a policy and a runtime adapter
3. Submit a turn that includes a planned tool call
4. Watch the event stream and understand each event

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make src/ importable

from src.core.orchestrator import Orchestrator
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType
from src.schemas.tool_io import RiskTier
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry

print("✓ imports ok")

### Step 1 — Register a tool

`ToolDescriptor` declares:
- `name` — how the model will refer to it
- `handler` — the actual Python callable (runs deterministically, never by the model)
- `risk_tier` — LOW / MEDIUM / HIGH / CRITICAL (drives mode selection)
- `is_state_changing` — `True` forces deterministic mode unconditionally

In [ ]:
registry = ToolRegistry()

registry.register(ToolDescriptor(
    name="my_tool",
    handler=lambda x: {"output": x * 2},   # doubles the input
    risk_tier=RiskTier.HIGH,
    is_state_changing=True,
))

print(f"✓ registered tools: {registry.list_tools()}")

### Step 2 — Wire the Orchestrator

Three components are injected:
- `runtime_adapter` — the provider boundary (currently a simulation stub)
- `policy_middleware` — `DeterministicFirstPolicyMiddleware` enforces deterministic execution
  for HIGH/CRITICAL and state-changing operations
- `tool_executor` — executes tools with the full decorator stack (validate → authz → retry → audit → redact)

In [ ]:
policy = DeterministicFirstPolicyMiddleware()

orchestrator = Orchestrator(
    runtime_adapter=OpenAIAgentsRuntimeAdapter(),
    policy_middleware=policy,
    tool_executor=DeterministicToolExecutor(registry=registry, policy=policy),
)

print("✓ orchestrator ready")
print(f"  adapter capabilities: {orchestrator._runtime_adapter.get_capabilities()}")

# verify healthcheck works (adapter is a simulation stub — always HEALTHY)
health = await orchestrator._runtime_adapter.healthcheck()
print(f"  adapter health:       {health.state.value}")

### Step 3 — Build the context and run a turn

The `context` dict carries session metadata and, crucially for this demo, a
`planned_tool_call`. In production, this comes from the model's response —
the adapter parses it and emits a `TOOL_INTENT` event. Here we inject it directly
to demonstrate the execution path without a live model.

**Event types you will see:**
| Event | Meaning |
|-------|---------|
| `TOOL_INTENT` | Adapter signalled that a tool should be called — orchestrator intercepts |
| `OUTPUT_DELTA` | Streamed text chunk from the model (or tool result acknowledgement) |
| `RUN_COMPLETE` | Turn finished — full output payload available |

In [ ]:
context = {
    "run_id":    "r1",
    "job_id":    "j1",
    "task_id":   "t1",
    "agent_id":  "a1",
    "planned_tool_call": {
        "call_id":          "tc1",
        "tool_name":        "my_tool",
        "arguments":        {"x": 5},
        "risk_tier":        RiskTier.HIGH.value,
        "is_state_changing": True,
    },
}

async def run_turn():
    events = []
    async for event in orchestrator.run_turn("sess1", "go", context):
        events.append(event)
        # print each event as it arrives
        print(f"  [{event.event_type.value:15s}]  payload={event.payload}")
    return events

print("── running turn ──────────────────────────────────────────")
events = await run_turn()
print("──────────────────────────────────────────────────────────")
print(f"\n✓ turn complete — {len(events)} events received")
types = {e.event_type for e in events}
assert RuntimeEventType.RUN_COMPLETE in types

### What just happened

```
1. Orchestrator called adapter.start_session()
2. Adapter.run_turn() read planned_tool_call → emitted TOOL_INTENT
3. Orchestrator intercepted TOOL_INTENT:
   a. PolicyMiddleware.before_tool_call()
      → risk=HIGH + is_state_changing=True → decision=ALLOW, mode=DETERMINISTIC
   b. ModeSelector confirmed DETERMINISTIC (state-changing overrides everything)
   c. DeterministicToolExecutor.execute()
      → validate args → authz check → call handler(x=5) → {"output": 10}
      → PolicyMiddleware.after_tool_call() (correlation_id + mode checks)
   d. ToolResult submitted back to adapter via submit_tool_results()
4. Adapter emitted OUTPUT_DELTA + RUN_COMPLETE
```

The tool `lambda x: {"output": x * 2}` with `x=5` returned `{"output": 10}`.
The model never executed the tool. The framework did — safely.

---
## Section 2 — Background DAG Execution

For long-running, multi-step workflows the framework provides a full background
job runtime built on a DAG scheduler.

```
BackgroundRuntime.submit(graph, payload, job_id)
      │
      ├── TenantQuotaManager.check_submission()   ← quota gate
      │
      └── TaskScheduler.execute(graph)
            │
            ├── Wave 1: all nodes with no dependencies run in parallel
            │     └── WorkerPool (bounded concurrency semaphore)
            │           └── _run_node()
            │                 ├── CheckpointStore.get()    ← resume if prior state
            │                 ├── handler(payload)         ← must be async def
            │                 ├── CheckpointStore.save()   ← durable progress
            │                 └── StructuredLogger / RuntimeMetrics / RuntimeTimeline
            │
            └── Wave 2…N: unlock nodes whose dependencies completed
```

**Critical rule:** every `TaskNode` handler must be `async def`.
The scheduler does `await asyncio.wait_for(handler(payload), timeout=…)`.
A plain `lambda` or sync function will silently fail with `TASK_EXECUTION_ERROR`.

In [ ]:
from src.core.background_runtime import BackgroundRuntime, JobStatus
from src.core.checkpoint_store import InMemoryCheckpointStore
from src.core.scheduler import TaskScheduler
from src.core.task_graph import TaskGraph, TaskNode
from src.core.worker_pool import WorkerPool
from src.observability.logging import StructuredLogger
from src.observability.metrics import RuntimeMetrics
from src.observability.timeline import RuntimeTimeline
import asyncio

print("✓ background runtime imports ok")

### Step 1 — Wire the runtime with observability

Three observability components are injected:
- `StructuredLogger` — structured log records with `correlation_id`, `level`, `event`, `context`
- `RuntimeMetrics` — named counters, latency observations, gauges
- `RuntimeTimeline` — append-only ordered event log per `correlation_id`

In [ ]:
logger   = StructuredLogger()
metrics  = RuntimeMetrics()
timeline = RuntimeTimeline()

scheduler = TaskScheduler(
    worker_pool=WorkerPool(max_concurrency=3),
    checkpoint_store=InMemoryCheckpointStore(),
    logger=logger,
    metrics=metrics,
    timeline=timeline,
)
runtime = BackgroundRuntime(
    scheduler=scheduler,
    logger=logger,
    metrics=metrics,
    timeline=timeline,
)

print("✓ BackgroundRuntime ready  (workers=3, checkpoint=in-memory)")

### Step 2 — Define the DAG

A two-node DAG where `process` depends on the output of `fetch`:

```
fetch ──► process
```

Each handler receives a `payload` dict that always contains:
- `payload["dependencies"]["<node_id>"]` — completed upstream node output
- `payload["node_id"]` — this node's id
- `payload["job_id"]` — the job id
- anything from the initial `payload` you passed to `submit()`

In [ ]:
async def fetch(payload: dict) -> dict:
    return {"data": 42}

async def process(payload: dict) -> dict:
    upstream = payload["dependencies"]["fetch"]["data"]
    return {"result": upstream * 2}

graph = TaskGraph([
    TaskNode("fetch",   handler=fetch),
    TaskNode("process", handler=process, depends_on=["fetch"]),
])

print("✓ graph defined")
print(f"  nodes: {graph.node_ids()}")

### Step 3 — Submit and wait

In [ ]:
async def run_dag():
    job_id = runtime.submit(graph=graph, payload={}, job_id="demo_job")
    print(f"submitted  job_id={job_id}  status={runtime.get_job(job_id).status.value}")

    # poll until done (max 2s)
    for i in range(200):
        status = runtime.get_job(job_id).status
        if status in {JobStatus.COMPLETED, JobStatus.FAILED}:
            print(f"done       iterations={i+1}  status={status.value}")
            break
        await asyncio.sleep(0.01)

    return runtime.get_job(job_id)

job = await run_dag()

### Step 4 — Inspect results, timeline, metrics, and logs

In [ ]:
job_id = "demo_job"

print("── node outcomes ─────────────────────────────────────────")
for node_id, outcome in job.result.outcomes.items():
    print(f"  {node_id:10s}  status={outcome.status.value:10s}  output={outcome.output}")

print("\n── metrics ───────────────────────────────────────────────")
for key, value in sorted(metrics.counters.items()):
    print(f"  {key} = {value}")

print("\n── timeline (ordered events for this job) ────────────────")
for entry in timeline.entries_for(job_id):
    print(f"  {entry.event:40s}  {entry.payload}")

print("\n── structured logs ───────────────────────────────────────")
for record in logger.records():
    if record.correlation_id == job_id:
        print(f"  [{record.level.value:5s}] {record.event:40s}  {record.context}")

assert job.status == JobStatus.COMPLETED
print("\n✓ PASS")

### What the output tells you

| What you see | What it means |
|---|---|
| `scheduler.node_started` | Scheduler began executing a node |
| `scheduler.node_completed` | Node handler returned successfully, checkpoint saved |
| `scheduler.job_completed` | All nodes finished (no failures) |
| `background.job_finished` | BackgroundRuntime marked job COMPLETED |
| `scheduler.node.success = 2` | 2 nodes completed across all jobs |
| `scheduler.queue_depth` | Remaining nodes at each wave boundary |

The `fetch` node output (`{"data": 42}`) flows into `process` via
`payload["dependencies"]["fetch"]`, which returns `{"result": 84}`.

---
## Summary — What "First Brick" gives you

| Capability | Status |
|---|---|
| Provider-neutral runtime contract | ✅ done |
| Deterministic-first policy enforcement | ✅ done |
| Risk-gate evaluation (ALLOW / DENY / ESCALATE) | ✅ done |
| RBAC / tenant overlay policy wiring | ✅ done |
| Background DAG scheduler with checkpoint/resume | ✅ done |
| Observability: structured logs, metrics, timeline, tracing | ✅ done |
| Persistence: session, checkpoint, event, audit, workflow stores | ✅ done |
| Resilience: retry, circuit breaker, DLQ, compensation hooks | ✅ done |
| Audit: tamper-evident SHA-256 hash chain | ✅ done |
| MCP server integration with trust tiers | ✅ done |

**What's missing:** a real provider adapter that calls an actual AI model.
That is what **Brick 2** builds.